# 简单高效 S 因子

BigAlpha 2026 AI 因子挖掘传统量化赛道提交。所有滚动统计只使用因子日及以前数据；main 返回且仅返回 date、instrument、factor 三列。


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from math import sqrt

import numpy as np
import pandas as pd


@dataclass(frozen=True)
class FastSConfig:
    half_life: float = 126.0
    horizon: int = 5
    min_state_observations: int = 30
    min_effective_observations: float = 15.0
    kappa: float = 20.0
    lambda_up: float = 0.25
    lambda_down: float = 0.50
    gamma: float = 1.0
    clip_sigma: float = 6.0

    def validate(self) -> None:
        positive = (
            self.half_life,
            self.horizon,
            self.min_state_observations,
            self.min_effective_observations,
            self.kappa,
            self.clip_sigma,
        )
        if any(value <= 0 for value in positive):
            raise ValueError("FastSConfig positive parameters must be positive")
        if self.lambda_up < 0 or self.lambda_down < 0 or self.gamma < 0:
            raise ValueError("S-value penalties cannot be negative")


SIMPLE_CONFIG = FastSConfig(half_life=126.0, horizon=5)
DEEP_CONFIGS = (
    FastSConfig(half_life=63.0, horizon=1),
    FastSConfig(half_life=126.0, horizon=5),
    FastSConfig(half_life=252.0, horizon=20),
)


def confirmed_ma_regime(
    benchmark_close: pd.Series,
    short_window: int = 5,
    long_window: int = 10,
    confirmation: int = 3,
) -> pd.Series:
    close = pd.to_numeric(benchmark_close, errors="coerce")
    short = close.rolling(short_window, min_periods=short_window).mean()
    long = close.rolling(long_window, min_periods=long_window).mean()
    raw = pd.Series(pd.NA, index=close.index, dtype="string")
    raw.loc[short.gt(long)] = "U"
    raw.loc[short.lt(long)] = "D"
    output = pd.Series(pd.NA, index=close.index, dtype="string")
    active: str | None = None
    candidate: str | None = None
    run = 0
    for position, value in enumerate(raw.array):
        current = None if pd.isna(value) else str(value)
        if current is None:
            candidate = None
            run = 0
        elif current == candidate:
            run += 1
        else:
            candidate = current
            run = 1
        if candidate is not None and run >= confirmation:
            active = candidate
        if active is not None:
            output.iloc[position] = active
    return output


def _normalize_inputs(
    bars: pd.DataFrame,
    benchmark: pd.DataFrame,
    components: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    bars = bars.copy()
    benchmark = benchmark.copy()
    components = components.copy()
    if "symbol" in bars and "instrument" not in bars:
        bars = bars.rename(columns={"symbol": "instrument"})
    if "symbol" in components and "instrument" not in components:
        components = components.rename(columns={"symbol": "instrument"})
    if "member_code" in components:
        components = components.rename(columns={"member_code": "instrument"})
    if "adj_close" in bars and "close" not in bars:
        bars = bars.rename(columns={"adj_close": "close"})
    if "adj_close" in benchmark and "close" not in benchmark:
        benchmark = benchmark.rename(columns={"adj_close": "close"})
    required_bars = {"date", "instrument", "close", "volume"}
    required_benchmark = {"date", "close"}
    required_components = {"date", "instrument"}
    for name, frame, required in (
        ("bars", bars, required_bars),
        ("benchmark", benchmark, required_benchmark),
        ("components", components, required_components),
    ):
        missing = sorted(required - set(frame.columns))
        if missing:
            raise ValueError(f"{name} is missing columns: {missing}")

    bars["date"] = pd.to_datetime(bars["date"], errors="raise").dt.normalize()
    benchmark["date"] = pd.to_datetime(benchmark["date"], errors="raise").dt.normalize()
    components["date"] = pd.to_datetime(components["date"], errors="raise").dt.normalize()
    bars["instrument"] = bars["instrument"].astype(str)
    components["instrument"] = components["instrument"].astype(str)
    bars = bars.drop_duplicates(["date", "instrument"], keep="last")
    components = components.drop_duplicates(["date", "instrument"], keep="last")
    benchmark = benchmark.drop_duplicates("date", keep="last").sort_values("date")
    market_close = benchmark.set_index("date")["close"].astype(float)
    calendar = pd.DatetimeIndex(market_close.index)
    bars = bars.loc[bars["date"].isin(calendar)].copy()
    components = components.loc[components["date"].isin(calendar)].copy()
    return bars, market_close, components


def _pivot_market_panel(
    bars: pd.DataFrame,
    calendar: pd.DatetimeIndex,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    close = bars.pivot(index="date", columns="instrument", values="close")
    volume = bars.pivot(index="date", columns="instrument", values="volume")
    symbols = sorted(set(close.columns).union(volume.columns))
    close = close.reindex(index=calendar, columns=symbols).astype(float)
    volume = volume.reindex(index=calendar, columns=symbols).astype(float)
    close = close.where(close.gt(0.0))
    volume = volume.where(volume.gt(0.0))
    return close, volume


def _state_q(
    weight: np.ndarray,
    squared_weight: np.ndarray,
    observations: np.ndarray,
    total: np.ndarray,
    total_square: np.ndarray,
    downside_square: np.ndarray,
    config: FastSConfig,
    penalty: float,
) -> np.ndarray:
    valid = (
        (weight > 0)
        & (observations >= config.min_state_observations)
        & (squared_weight > 0)
    )
    n_eff = np.divide(
        weight * weight,
        squared_weight,
        out=np.full_like(weight, np.nan),
        where=squared_weight > 0,
    )
    mean = np.divide(total, weight, out=np.zeros_like(total), where=weight > 0)
    second = np.divide(
        total_square, weight, out=np.zeros_like(total_square), where=weight > 0
    )
    variance = np.maximum(second - mean * mean, 0.0)
    downside = np.sqrt(
        np.maximum(
            np.divide(
                downside_square,
                weight,
                out=np.zeros_like(downside_square),
                where=weight > 0,
            ),
            0.0,
        )
    )
    standard_error = np.sqrt(variance) / np.sqrt(np.maximum(n_eff, 1.0))
    shrunk = n_eff / (n_eff + config.kappa) * mean
    q = (
        config.horizon * shrunk
        - penalty * sqrt(config.horizon) * downside
        - config.gamma * config.horizon * standard_error
    )
    q[~valid | (n_eff < config.min_effective_observations)] = np.nan
    return q


def _run_one_scale(
    close: pd.DataFrame,
    volume: pd.DataFrame,
    market_close: pd.Series,
    regimes: pd.Series,
    config: FastSConfig,
) -> dict[str, pd.DataFrame]:
    """Run a causal O(trading days x stocks) recursive S approximation."""

    config.validate()
    dates = close.index
    symbols = close.columns
    prices = close.to_numpy(dtype=float)
    traded = volume.notna().to_numpy(dtype=bool) & np.isfinite(prices)
    stock_returns = np.full_like(prices, np.nan)
    valid_pair = traded[1:] & traded[:-1]
    ratio = np.divide(
        prices[1:], prices[:-1], out=np.full_like(prices[1:], np.nan), where=valid_pair
    )
    stock_returns[1:] = np.where(valid_pair & (ratio > 0), np.log(ratio), np.nan)
    market_returns = np.log(market_close / market_close.shift(1)).to_numpy(dtype=float)

    size = len(symbols)
    decay = 2.0 ** (-1.0 / config.half_life)
    decay_square = decay * decay

    beta_w = {state: np.zeros(size) for state in ("U", "D")}
    beta_x = {state: np.zeros(size) for state in ("U", "D")}
    beta_m = {state: np.zeros(size) for state in ("U", "D")}
    beta_xm = {state: np.zeros(size) for state in ("U", "D")}
    beta_mm = {state: np.zeros(size) for state in ("U", "D")}
    alpha_w = {state: np.zeros(size) for state in ("U", "D")}
    alpha_w2 = {state: np.zeros(size) for state in ("U", "D")}
    alpha_n = {state: np.zeros(size, dtype=np.int64) for state in ("U", "D")}
    alpha_x = {state: np.zeros(size) for state in ("U", "D")}
    alpha_x2 = {state: np.zeros(size) for state in ("U", "D")}
    alpha_down2 = {state: np.zeros(size) for state in ("U", "D")}
    robust_w = np.zeros(size)
    robust_x = np.zeros(size)
    robust_x2 = np.zeros(size)

    s_values = np.full_like(prices, np.nan)
    q_up_values = np.full_like(prices, np.nan)
    q_down_values = np.full_like(prices, np.nan)
    for position in range(len(dates)):
        # Emit today's factor before updating with today's close-to-close return.
        # This makes the date-t factor depend only on information available up to t-1.
        q_up = _state_q(
            alpha_w["U"],
            alpha_w2["U"],
            alpha_n["U"],
            alpha_x["U"],
            alpha_x2["U"],
            alpha_down2["U"],
            config,
            config.lambda_up,
        )
        q_down = _state_q(
            alpha_w["D"],
            alpha_w2["D"],
            alpha_n["D"],
            alpha_x["D"],
            alpha_x2["D"],
            alpha_down2["D"],
            config,
            config.lambda_down,
        )
        q_up_values[position] = q_up
        q_down_values[position] = q_down
        s_values[position] = 0.5 * (q_up + q_down)

        for state in ("U", "D"):
            beta_w[state] *= decay
            beta_x[state] *= decay
            beta_m[state] *= decay
            beta_xm[state] *= decay
            beta_mm[state] *= decay
            alpha_w[state] *= decay
            alpha_w2[state] *= decay_square
            alpha_x[state] *= decay
            alpha_x2[state] *= decay
            alpha_down2[state] *= decay
        robust_w *= decay
        robust_x *= decay
        robust_x2 *= decay

        state_value = regimes.iloc[position - 1] if position > 0 else pd.NA
        state = None if pd.isna(state_value) else str(state_value)
        market_return = market_returns[position]
        current = stock_returns[position]
        valid = np.isfinite(current) & np.isfinite(market_return)
        if state in ("U", "D") and valid.any():
            weight = beta_w[state]
            mean_x = np.divide(beta_x[state], weight, out=np.zeros(size), where=weight > 0)
            mean_m = np.divide(beta_m[state], weight, out=np.zeros(size), where=weight > 0)
            covariance = beta_xm[state] - weight * mean_x * mean_m
            market_variance = beta_mm[state] - weight * mean_m * mean_m
            beta = np.divide(
                covariance,
                market_variance,
                out=np.ones(size),
                where=market_variance > 1e-16,
            )
            residual = current - beta * market_return
            robust_mean = np.divide(
                robust_x, robust_w, out=np.zeros(size), where=robust_w > 0
            )
            robust_var = np.maximum(
                np.divide(robust_x2, robust_w, out=np.zeros(size), where=robust_w > 0)
                - robust_mean * robust_mean,
                0.0,
            )
            boundary = config.clip_sigma * np.sqrt(robust_var)
            can_clip = robust_w >= 20
            clipped = residual.copy()
            clipped[can_clip] = np.clip(
                clipped[can_clip], -boundary[can_clip], boundary[can_clip]
            )

            v = valid.astype(float)
            x = np.where(valid, current, 0.0)
            m = float(market_return)
            r = np.where(valid, clipped, 0.0)
            beta_w[state] += v
            beta_x[state] += x
            beta_m[state] += v * m
            beta_xm[state] += x * m
            beta_mm[state] += v * m * m
            alpha_w[state] += v
            alpha_w2[state] += v
            alpha_n[state] += valid.astype(np.int64)
            alpha_x[state] += r
            alpha_x2[state] += r * r
            alpha_down2[state] += np.minimum(r, 0.0) ** 2
            robust_w += v
            robust_x += r
            robust_x2 += r * r

    return {
        "s": pd.DataFrame(s_values, index=dates, columns=symbols),
        "q_up": pd.DataFrame(q_up_values, index=dates, columns=symbols),
        "q_down": pd.DataFrame(q_down_values, index=dates, columns=symbols),
        "returns": pd.DataFrame(stock_returns, index=dates, columns=symbols),
    }


def _rank_panel(frame: pd.DataFrame) -> pd.DataFrame:
    ranks = frame.rank(axis=1, method="average", pct=True)
    return 2.0 * ranks - 1.0


def _membership_panel(
    components: pd.DataFrame,
    calendar: pd.DatetimeIndex,
    symbols: pd.Index,
) -> pd.DataFrame:
    """Return the point-in-time index universe for each trading day."""

    membership = (
        components.assign(_member=True)
        .pivot(index="date", columns="instrument", values="_member")
        .reindex(index=calendar, columns=symbols)
    )
    return membership.fillna(False).astype(bool)


def _style_neutralize(
    target: pd.DataFrame,
    exposures: list[pd.DataFrame],
    ridge: float = 1e-3,
) -> pd.DataFrame:
    output = pd.DataFrame(np.nan, index=target.index, columns=target.columns)
    ranked_exposures = [_rank_panel(frame) for frame in exposures]
    for position in range(len(target.index)):
        y = target.iloc[position].to_numpy(dtype=float)
        x_columns = [frame.iloc[position].to_numpy(dtype=float) for frame in ranked_exposures]
        x = np.column_stack(x_columns)
        valid = np.isfinite(y) & np.isfinite(x).all(axis=1)
        if valid.sum() <= x.shape[1] + 5:
            output.iloc[position] = y
            continue
        design = np.column_stack([np.ones(valid.sum()), x[valid]])
        penalty = np.eye(design.shape[1]) * ridge
        penalty[0, 0] = 0.0
        coefficients = np.linalg.solve(
            design.T @ design + penalty, design.T @ y[valid]
        )
        residual = np.full_like(y, np.nan)
        residual[valid] = y[valid] - design @ coefficients
        output.iloc[position] = residual
    return output


def _finalize_membership_factor(
    factor_panel: pd.DataFrame,
    components: pd.DataFrame,
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    # Missing panel cells are restored by the membership merge below, so the
    # default stack behavior works on both pandas 2.x and 3.x.
    long = factor_panel.stack().rename("factor").reset_index()
    long.columns = ["date", "instrument", "factor"]
    output = components.merge(long, on=["date", "instrument"], how="left", validate="one_to_one")
    output = output.loc[
        output["date"].between(pd.Timestamp(start_date), pd.Timestamp(end_date))
    ].copy()
    output["factor"] = output.groupby("date")["factor"].transform(
        lambda values: values.fillna(values.median())
    )
    output["factor"] = output["factor"].fillna(0.0).astype(float)
    if output.duplicated(["date", "instrument"]).any():
        raise ValueError("factor output contains duplicate date/instrument keys")
    if not np.isfinite(output["factor"]).all():
        raise ValueError("factor output contains non-finite values")
    return output[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)


def compute_svalue_factor(
    bars: pd.DataFrame,
    benchmark: pd.DataFrame,
    components: pd.DataFrame,
    start_date: str,
    end_date: str,
    mode: str = "simple",
) -> pd.DataFrame:
    """Compute a competition-ready daily S factor with no future inputs.

    `simple` uses one robust state-conditioned S scale. `deep` combines three
    causal scales and removes daily linear exposure to momentum, volatility,
    beta-like market sensitivity, and liquidity proxies.
    """

    if mode not in {"simple", "deep"}:
        raise ValueError("mode must be 'simple' or 'deep'")
    bars, market_close, components = _normalize_inputs(bars, benchmark, components)
    calendar = pd.DatetimeIndex(market_close.index)
    close, volume = _pivot_market_panel(bars, calendar)
    membership = _membership_panel(components, calendar, close.columns)
    regimes = confirmed_ma_regime(market_close)

    if mode == "simple":
        result = _run_one_scale(close, volume, market_close, regimes, SIMPLE_CONFIG)
        # Rank only today's members. Ranking over the full-period union would
        # let stocks that join the index in the future alter historical scores.
        factor = _rank_panel(result["s"].where(membership))
    else:
        scale_results = [
            _run_one_scale(close, volume, market_close, regimes, config)
            for config in DEEP_CONFIGS
        ]
        ranked_s = [
            _rank_panel(result["s"].where(membership))
            for result in scale_results
        ]
        worst_state = pd.DataFrame(
            np.minimum(scale_results[1]["q_up"], scale_results[1]["q_down"]),
            index=close.index,
            columns=close.columns,
        ).where(membership)
        state_fragility = (
            scale_results[1]["q_up"] - scale_results[1]["q_down"]
        ).abs().where(membership)
        raw = (
            0.20 * ranked_s[0]
            + 0.45 * ranked_s[1]
            + 0.20 * ranked_s[2]
            + 0.10 * _rank_panel(worst_state)
            + 0.05 * _rank_panel(-state_fragility)
        )
        returns = scale_results[1]["returns"]
        # The date-t factor is formed before the date-t close. Style controls
        # must therefore be known at t-1 as well.
        momentum_20 = np.log(close / close.shift(20)).shift(1)
        momentum_126 = np.log(close / close.shift(126)).shift(1)
        volatility_60 = returns.rolling(60, min_periods=30).std().shift(1)
        liquidity_20 = (
            np.log((close * volume).rolling(20, min_periods=10).mean()).shift(1)
        )
        market_return = np.log(market_close / market_close.shift(1))
        beta_126 = (
            returns.rolling(126, min_periods=60)
            .cov(market_return)
            .divide(market_return.rolling(126, min_periods=60).var(), axis=0)
            .shift(1)
        )
        neutral = _style_neutralize(
            raw,
            [
                momentum_20.where(membership),
                momentum_126.where(membership),
                volatility_60.where(membership),
                beta_126.where(membership),
                liquidity_20.where(membership),
            ],
        )
        factor = _rank_panel(neutral)
    return _finalize_membership_factor(factor, components, start_date, end_date)


from datetime import date, timedelta

import pandas as pd


COMPETITION_INDEX = "000852.SH"
OUTPUT_START = "2019-01-01"
OUTPUT_END = "2024-12-31"
QUERY_START = "2017-01-01"
QUERY_END_EXCLUSIVE = "2025-01-01"


def _sql_string(value: str) -> str:
    return "'" + str(value).replace("'", "''") + "'"


def _chunks(values: list[str], size: int) -> list[list[str]]:
    return [values[position : position + size] for position in range(0, len(values), size)]


def load_bigalpha_daily_data(start_date: str = OUTPUT_START, end_date: str = OUTPUT_END) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load bounded competition inputs inside BigQuant AIStudio.

    The stock query is split by instrument so the DAI request remains bounded.
    All queries use an explicit half-open date filter; no external network is
    needed or used.
    """

    import dai

    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=730)).strftime("%Y-%m-%d")
    query_end_exclusive = (pd.to_datetime(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    date_filter = {"date": [query_start, query_end_exclusive]}
    component_sql = f"""
        SELECT date, member_code AS instrument
        FROM cn_stock_index_component
        WHERE date >= {_sql_string(query_start)}
          AND date < {_sql_string(query_end_exclusive)}
          AND instrument = {_sql_string(COMPETITION_INDEX)}
        ORDER BY date, member_code
    """
    components = dai.query(component_sql, filters=date_filter).df()
    if components.empty:
        raise RuntimeError("CSI 1000 component query returned no rows")
    components["instrument"] = components["instrument"].astype(str)
    symbols = sorted(components["instrument"].unique().tolist())

    bar_parts: list[pd.DataFrame] = []
    for symbol_chunk in _chunks(symbols, 250):
        values = ", ".join(_sql_string(symbol) for symbol in symbol_chunk)
        bar_sql = f"""
            SELECT date, instrument, close, volume
            FROM cn_stock_bar1d
            WHERE date >= {_sql_string(query_start)}
              AND date < {_sql_string(query_end_exclusive)}
              AND instrument IN ({values})
            ORDER BY date, instrument
        """
        bar_parts.append(dai.query(bar_sql, filters=date_filter).df())
    bars = pd.concat(bar_parts, ignore_index=True)
    if bars.empty:
        raise RuntimeError("stock bar query returned no rows")

    benchmark_sql = f"""
        SELECT date, instrument, close
        FROM cn_stock_index_bar1d
        WHERE date >= {_sql_string(query_start)}
          AND date < {_sql_string(query_end_exclusive)}
          AND instrument = {_sql_string(COMPETITION_INDEX)}
        ORDER BY date
    """
    benchmark = dai.query(benchmark_sql, filters=date_filter).df()
    if benchmark.empty:
        raise RuntimeError("CSI 1000 index bar query returned no rows")
    return bars, benchmark, components


def audit_competition_output(
    result: pd.DataFrame,
    components: pd.DataFrame,
    start_date: str = OUTPUT_START,
    end_date: str = OUTPUT_END,
) -> None:
    expected_columns = ["date", "instrument", "factor"]
    if result.columns.tolist() != expected_columns:
        raise ValueError(f"factor columns must be exactly {expected_columns}")
    if result.duplicated(["date", "instrument"]).any():
        raise ValueError("factor output has duplicate keys")
    output_dates = pd.DatetimeIndex(pd.to_datetime(result["date"]).unique()).sort_values()
    expected_dates = pd.DatetimeIndex(
        pd.to_datetime(
            components.loc[
                pd.to_datetime(components["date"]).between(start_date, end_date), "date"
            ].unique()
        )
    ).sort_values()
    if not output_dates.equals(expected_dates):
        raise ValueError("factor output is missing one or more competition trading days")
    daily_missing = result.groupby("date")["factor"].apply(lambda values: values.isna().mean())
    if (daily_missing > 0.40).any():
        raise ValueError("daily factor missing rate exceeds 40%")


def main(datasources=None, start_date: str = OUTPUT_START, end_date: str = OUTPUT_END):
    """BigAlpha traditional-track submission: causal multi-scale S factor.

    The platform calls this as main(datasources, start_date, end_date). The
    datasources mapping is accepted for template compatibility; this daily S
    implementation uses BigQuant daily tables and the injected scoring window.
    """

    output_start = pd.to_datetime(start_date).strftime("%Y-%m-%d")
    output_end = pd.to_datetime(end_date).strftime("%Y-%m-%d")
    bars, benchmark, components = load_bigalpha_daily_data(output_start, output_end)
    result = compute_svalue_factor(
        bars,
        benchmark,
        components,
        start_date=output_start,
        end_date=output_end,
        mode="deep",
    )
    audit_competition_output(result, components, output_start, output_end)
    return result


Platform runner calls `main(datasources, start_date, end_date)`.
